In [1]:
!pip install tensorflow nltk

In [2]:
import numpy as np
import nltk
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
from nltk.corpus import gutenberg
nltk.download('gutenberg')

text = gutenberg.raw('shakespeare-hamlet.txt')
print(f"Original text length: {len(text)} characters")

text = text.lower()
text = re.sub(r'[^a-z\s]', '', text)
print(f"Cleaned text length: {len(text)} characters")

Original text length: 162881 characters
Cleaned text length: 155780 characters


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Soham\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print(f"Total unique words in vocabulary: {len(tokenizer.word_index)}")
print(f"Vocabulary Size (incl. OOV): {total_words}")

Total unique words in vocabulary: 4796
Vocabulary Size (incl. OOV): 4797


In [5]:
input_sequences = []

words = text.split()
sequence_length = 5
print(f"Sequence length: {sequence_length} words")
print(f"Total words in text: {len(words)}")

for i in range(sequence_length, len(words)):
    seq = words[i-sequence_length:i+1]
    token_list = tokenizer.texts_to_sequences([' '.join(seq)])[0]
    input_sequences.append(token_list)

input_sequences = np.array(input_sequences)
print(f"Total sequences created: {input_sequences.shape[0]}")
print(f"Shape of sequences: {input_sequences.shape}")

Sequence length: 5 words
Total words in text: 29578
Total sequences created: 29573
Shape of sequences: (29573, 6)


In [6]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print(f"X shape (inputs): {X.shape}")
print(f"y shape before one-hot encoding: {y.shape}")

from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=total_words)
print(f"y shape after one-hot encoding: {y.shape}")
print(f"One-hot encoding creates a vector of size {total_words} for each target word")

X shape (inputs): (29573, 5)
y shape before one-hot encoding: (29573,)
y shape after one-hot encoding: (29573, 4797)
One-hot encoding creates a vector of size 4797 for each target word


In [7]:
model = Sequential([
    Embedding(total_words, 100, input_length=sequence_length),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

c:\Users\Soham\Codes\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(
    X, y,
    epochs=30,
    verbose=1,
    batch_size=32
)
print("\nTraining completed!")

Epoch 1/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.0325 - loss: 6.8121
Epoch 2/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.0437 - loss: 6.3816
Epoch 3/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.0588 - loss: 6.1334
Epoch 4/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.0764 - loss: 5.8559
Epoch 5/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.0916 - loss: 5.5717
Epoch 6/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.1041 - loss: 5.2600
Epoch 7/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.1203 - loss: 4.9085
Epoch 8/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.1502 - loss: 4.5346
Epoch 9/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.1960 - loss: 4.1586
Epoch 10/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.2561 - loss: 3.7799
Epoch 11/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.3167 - loss: 3.4174
Epoch 12/30
925/925 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/ste

In [9]:
def generate_text(seed_text, next_words=10):
    print(f"\nGenerating {next_words} words from: '{seed_text}'")
    
    for iteration in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=sequence_length, padding='pre')
        
        prediction_probs = model.predict(token_list, verbose=0)
        predicted_idx = np.argmax(prediction_probs)
        predicted_prob = prediction_probs[0][predicted_idx]
        
        predicted_word = None
        for word, index in tokenizer.word_index.items():
            if index == predicted_idx:
                predicted_word = word
                break
        
        if predicted_word:
            seed_text += " " + predicted_word
            print(f"  Step {iteration+1}: Predicted '{predicted_word}' (confidence: {predicted_prob:.2%})")
                
    return seed_text

In [10]:
generated_text_1 = generate_text("to be or not", 10)
print(f"\nGenerated: {generated_text_1}")

generated_text_2 = generate_text("the king is", 10)
print(f"\nGenerated: {generated_text_2}")


Generating 10 words from: 'to be or not'
  Step 1: Predicted 'common' (confidence: 18.74%)
  Step 2: Predicted 'time' (confidence: 14.35%)
  Step 3: Predicted 'with' (confidence: 63.85%)
  Step 4: Predicted 'you' (confidence: 86.00%)
  Step 5: Predicted 'for' (confidence: 83.55%)
  Step 6: Predicted 'all' (confidence: 46.29%)
  Step 7: Predicted 'i' (confidence: 58.00%)
  Step 8: Predicted 'know' (confidence: 46.07%)
  Step 9: Predicted 'my' (confidence: 50.17%)
  Step 10: Predicted 'daughter' (confidence: 22.41%)

Generated: to be or not common time with you for all i know my daughter

Generating 10 words from: 'the king is'
  Step 1: Predicted 'the' (confidence: 66.48%)
  Step 2: Predicted 'king' (confidence: 65.69%)
  Step 3: Predicted 'i' (confidence: 41.50%)
  Step 4: Predicted 'saw' (confidence: 44.43%)
  Step 5: Predicted 'the' (confidence: 61.41%)
  Step 6: Predicted 'cherube' (confidence: 58.45%)
  Step 7: Predicted 'that' (confidence: 70.53%)
  Step 8: Predicted 'sees' (conf